In [1]:
from cirq_sic import *

* Test the four tasks.
* Implement entangle and don't measure.
* Write data processor: metrics and images. Gate counts.
* 1-parameter family of experiments -> metrices and images.
* Runner.
* Get arbitrary d going. [Compare aux performance].
* Negativity measure [Identify barycenters...]
* Document.
* [Time evolution.]

In [2]:
specs = {"dataset_id": "test",
         "processor_id": "willow_pink",
         "run_type": "clean",
         "qubits": get_wh_qubits(2, "ak"),
         "n_shots": 50000,
         "optimizer": "cirq",
         "d": 2,
         "fiducial": rand_ket(2),
         "fiducial_description": "rand_ket",
         "wh_implementation": "ak"}

In [3]:
for task_type in sk_ground_tasks:
    task = task_from_specs(task_type, specs)
    run_sky_ground_task(task)

2025-10-22 04:16:20 [INFO] test/d2/CharacterizeWHReferenceDeviceTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Task already exists. Skipping.
2025-10-22 04:16:20 [INFO] test/d2/WHPOVMOnBasisStatesTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Task already exists. Skipping.
2025-10-22 04:16:20 [INFO] test/d2/BasisMeasurementOnWHStatesTask/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Task already exists. Skipping.
2025-10-22 04:16:20 [INFO] test/d2/BasisMeasurementOnBasisStatesTask/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Task already exists. Skipping.


In [4]:
tasks, sg_results = load_sky_ground_results(specs, separate=True)

In [7]:
task = tasks[CharacterizeWHReferenceDeviceTask]
E = wh_povm(np.array(task.fiducial))
P = np.array([[(a@b).trace()/b.trace() for b in E] for a in E]).real
assert np.allclose(P, exactify(task)["P"])
assert np.linalg.norm(sg_results["P"] - P) < 1e-2

In [8]:
task = tasks[WHPOVMOnBasisStatesTask]
E = wh_povm(np.array(task.fiducial))
Pi = [np.diag(np.eye(task.d)[i]) for i in range(task.d)]
p = np.array([[(a@b).trace() for b in Pi] for a in E]).real
assert np.allclose(p, exactify(task)["p"])
assert np.linalg.norm(sg_results["p"] - p) < 1e-2

In [9]:
task = tasks[BasisMeasurementOnWHStatesTask]
E = wh_povm(np.array(task.fiducial))
Pi = [np.diag(np.eye(task.d)[i]) for i in range(task.d)]
C = np.array([[(a@b).trace()/b.trace() for b in E] for a in Pi]).real
assert np.allclose(C, exactify(task)["C"])
assert np.linalg.norm(sg_results["C"] - C) < 1e-2

In [10]:
task = tasks[BasisMeasurementOnBasisStatesTask]
q = np.eye(task.d)
assert np.allclose(q, exactify(task)["q"])
assert np.linalg.norm(sg_results["q"] - q) < 1e-2